Dicom fastreader is modified from:
https://www.kaggle.com/code/nartaa/rsna-knee-0949-anatomical-mirror   
- Full credits go to @nartaa (https://www.kaggle.com/nartaa)

image encoder is MaVIT, from : https://github.com/qhfan/MALA  
- [paper] Rectifying Magnitude Neglect in Linear Attention - ICCV 2025, Qihang Fan

In [ ]:
import sys
sys.path.append("/kaggle/input/datasets/hengck23/hengck23-rnsa-knee")


import numpy as np
import torch
import pandas as pd
import time

from sklearn.metrics import roc_auc_score

from model import *
from my_fastread_knee import *


class DotDict(dict):
    def __getattr__(self, key):
        try:
            return self[key]
        except KeyError:
            raise AttributeError(key)

    def __setattr__(self, key, value):
        self[key] = value

    def __delattr__(self, key):
        try:
            del self[key]
        except KeyError:
            raise AttributeError(key)


print ("import OK!!!")

In [ ]:
MODE = "submit" #submit local

#-----------------------------

LABEL = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA',
          'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']

NUM_VIEW   = 6
NUM_SLICE  = 8
NUM_INJURY = 12
SIZE = 288
DEVICE = "cuda"
CHECKPOINT ="/kaggle/input/datasets/hengck23/hengck23-rnsa-knee/00000008.pth"

KAGGLE_DIR = "/kaggle/input/competitions/rsna-knee-abnormality-detection"

if MODE=="local":
    csv_dir = "/kaggle/input/datasets/hengck23/hengck23-rnsa-knee"
    DICOM_DIR = f"{KAGGLE_DIR}/train_series"
    STUDY = pd.read_csv(f"{csv_dir}/train_golden58.csv")
    SERIES = pd.read_csv(f"{csv_dir}/train_golden58_series.csv")
    truth = STUDY[LABEL].values
    
if MODE=="submit":
    DICOM_DIR = f"{KAGGLE_DIR}/test_series"
    STUDY = pd.read_csv(f"{KAGGLE_DIR}/sample_submission.csv")
    SERIES = pd.read_csv(f"{KAGGLE_DIR}/test_series.csv")



STUDY_ID = STUDY["StudyInstanceUID"].to_list()
print("STUDY_ID", len(STUDY_ID))

print ("config OK!!!")

In [ ]:
model = KneeModel(pretrained=False) 
saved = torch.load(CHECKPOINT, map_location=DEVICE, weights_only=False)
print(
    "resume model:",
    model.load_state_dict(saved["model_state_dict"], strict=False),
)
model.to(DEVICE)
model = model.eval()

fastreader = make_knee_dataset_reader(
    out_size=SIZE, num_slice=NUM_SLICE,
    percentile=(1.0, 99.0), crop_mm=130.0,
    io_threads=32,
    crop_only_if_nonsquare=True,
)


predict=[]
start_time = time.time()
for i,study_id in enumerate(STUDY_ID):
    if i<30: print(i, study_id,"--------------------")
    series = SERIES[SERIES["StudyInstanceUID"] == study_id]
    try:
        r = fastreader(study_id, series, DICOM_DIR, "valid")
        volume = torch.from_numpy(r["volume"]).unsqueeze(0).byte().to(DEVICE)
        view_mask = torch.from_numpy(r["view_mask"]).unsqueeze(0).bool().to(DEVICE)
        slice = torch.from_numpy(r["slice"]).unsqueeze(0).float().to(DEVICE)
    
        #print(volume.shape)
        # print(view_mask.shape)
        # print(slice.shape)
    
        with torch.no_grad():
            with torch.amp.autocast("cuda", dtype=torch.float16):
                logit = model.infer( volume, view_mask, slice)
                prob = torch.sigmoid(logit)
        prob = prob.detach().float().cpu().numpy()
    except:
        prob = np.zeros((1,12),np.float32)
        
    predict.append(prob)
    elapsed = time.time() - start_time
    if i<30: print(f"time: {elapsed/(i+1): 5.3f} sec/study")
predict = np.concat(predict, 0)

submit_df = pd.DataFrame(predict, columns=LABEL, index=STUDY_ID)
submit_df.index.name = "StudyInstanceUID"
submit_df.to_csv(f"submission.csv", index=True)


print(submit_df.head(5))
print("submit_df:", submit_df.shape)
print("MODE:", MODE)
!ls
print("Submit OK!!!!")

In [ ]:
if MODE=="local":
    
    mean_auc  = 0
    for i in range(len(LABEL)):
        t = truth[:, i]
        p = predict[:, i]
        auc = float(roc_auc_score(t, p))
        mean_auc += auc
        print(f"{LABEL[i]:20s}: {auc:.4f}")
    mean_auc = mean_auc /12
    print(f"MEAN AUC: {mean_auc:.4f}")